# Executar produtor Kafka — Indicadores Municipio
Tech Challenge Fase 2 - Pipeline Híbrida de Alfabetização

Este notebook instala a dependência, lê as credenciais do Confluent Cloud via **Databricks Secrets** e executa o script `producer_municipio.py`.

## 1. Instalar a biblioteca `confluent-kafka`

In [0]:
%pip install confluent-kafka --break-system-packages

In [0]:
dbutils.library.restartPython()

In [0]:
import yaml
import pathlib
import os
import subprocess

## 2. Carregar configurações do `config_municipio.yaml`
Os parâmetros `script_path`, `topic`, `intervalo` e `quantidade` são lidos do arquivo **`config_municipio.yaml`** no mesmo diretório deste notebook.

As credenciais do Confluent Cloud são lidas automaticamente do **Databricks Secrets** (scope `tc_02`).

In [0]:
_config_path = pathlib.Path("config_municipio.yaml")
if not _config_path.exists():
    raise FileNotFoundError(
        f"Arquivo de configuração não encontrado: {_config_path.resolve()}. "
        "Envie 'config_municipio.yaml' para o mesmo diretório deste notebook no Workspace."
    )

with open(_config_path) as _f:
    cfg = yaml.safe_load(_f)

print("Configuração carregada:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

In [0]:
SECRET_SCOPE = "tc_02"

os.environ["CONFLUENT_BOOTSTRAP_SERVERS"] = dbutils.secrets.get(scope=SECRET_SCOPE, key="bootstrap_servers")
os.environ["CONFLUENT_API_KEY"]            = dbutils.secrets.get(scope=SECRET_SCOPE, key="api_key")
os.environ["CONFLUENT_API_SECRET"]         = dbutils.secrets.get(scope=SECRET_SCOPE, key="api_secret")
os.environ["KAFKA_TOPIC"]                  = cfg["topic"]

script_path = cfg["script_path"]
if not os.path.exists(script_path):
    raise FileNotFoundError(
        f"Script não encontrado em '{script_path}'. "
        "Confira 'script_path' no config_municipio.yaml e se o arquivo foi enviado ao workspace."
    )

print("Credenciais carregadas do scope:", SECRET_SCOPE)
print("Tópico:", os.environ["KAFKA_TOPIC"])
print("Script localizado em:", script_path)

## 3. Executar o produtor
Equivalente a rodar `python producer_municipio.py --intervalo 3 --quantidade N` no terminal, mas passando o ambiente (`os.environ`) explicitamente para o subprocesso — assim as credenciais chegam ao script mesmo sem `export` persistir entre células.

In [0]:
comando = [
    "python", script_path,
    "--intervalo", str(cfg["intervalo"]),
    "--quantidade", str(cfg["quantidade"]),
]

processo = subprocess.Popen(
    comando,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for linha in processo.stdout:
    print(linha, end="")

processo.wait()
print(f"\nProcesso finalizado com código de saída {processo.returncode}")